# Material interfaces and skeletal partitions

The published PDE and macro configurations of Chaumont-Frelet, Paredes and Valentin (2026), DOI [10.1016/j.camwa.2026.01.016](https://doi.org/10.1016/j.camwa.2026.01.016), separate material integration, local approximation and skeletal approximation. The recorded local P4 meshes are stated explicitly; their equality to the unspecified historical local mesh is not assumed.

The main equations execute from source and state their current discretization. The original archived comparisons below execute with `--historical` and require their exact attributed payloads. `--study` runs the complete public current acquisition declared in the catalogue, with its own output provenance.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/a7dabfe7e919784f4399e189a02b98fad58ba33767f15a9c1f3baee4415d4639/38_unfitted_darcy-companion.zip"
COMPANION_SHA256 = "a7dabfe7e919784f4399e189a02b98fad58ba33767f15a9c1f3baee4415d4639"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("darcy/38_unfitted_darcy.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


import os

HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
STUDY_REPLAY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"


In [ ]:
import json
from pathlib import Path
import numpy as np
from IPython.display import Image, display
from threadpoolctl import threadpool_limits
from pymhm import FaceSpace, SkeletonSpace, TriangleMesh
from pymhm.fem.quadrature.material import fit_material_faces, fit_material_mesh
from pymhm.io.reservoir import CartesianCellField
from pymhm.adaptivity.darcy import solve_adaptive_darcy


## A local transmission patch

With $a=1$ to the left of $x=1/2$ and $a=7$ to the right, $p=\min(x,1/2)+\max(x-1/2,0)/7$ has a pressure-gradient jump but constant physical flux $q=(-1,0)$. The initial fine mesh crosses the interface. Fitting both the local mesh and integration partition gives a direct check of the transmission convention.

## Declare the local and global equations

The application writes local A, B and C blocks with the shared FEM kernels,
then supplies them to the generic assembler. Primal coefficients satisfy

$$
\begin{aligned}
a_T(p,v)&=(K\nabla p,\nabla v)_T,\\
L_T(v)&=(f,v)_T+\sum_j\alpha_jv(x_j),\\
B_T\lambda&=\langle\lambda,v\rangle_{\partial T},\\
C_Tp&=-\langle p,\mu\rangle_{\partial T}.
\end{aligned}
$$

Signs follow the fixed macroface normals. RT0 instead uses flux, pressure and
private boundary-pressure coefficients, including the local normal-flux
constraint. The global `Equation` supplies the signed Dirichlet moments and
retains each constant pressure mode. Pure Neumann data add the physical pressure
integral through an explicit constraint. `define_darcy` returns equations;
`assemble`, the global solve and field interpretation are separate stages.


In [ ]:
from pymhm import assemble, SolverConfig
from examples.formulations.darcy import define_darcy, pressure_constraints, recover_darcy

macro = TriangleMesh.unit_square()
a = CartesianCellField(np.array([[1.0], [7.0]]), (0.5, 1.0))
sk = fit_material_faces(SkeletonSpace(macro), a)
locals_ = tuple((fit_material_mesh(macro.submesh(i, 3), a) for i in range(2)))
exact = lambda x: np.minimum(x[:, 0], 0.5) + np.maximum(x[:, 0] - 0.5, 0) / 7
with threadpool_limits(1):
    patch_definition = define_darcy(
        macro,
        permeability=a,
        dirichlet=exact,
        local_meshes=locals_,
        skeleton=sk,
        quadrature_order=6,
    )
    patch_system = assemble(patch_definition.problem)
    patch = recover_darcy(
        patch_definition,
        patch_system,
        patch_system.solve(constraints=pressure_constraints(patch_definition, patch_system)),
    )
print(
    {
        "pressure_L2_error": patch.l2_error(exact),
        "physical_flux_L2_error": patch.flux_l2_error((-1.0, 0.0)),
    }
)


## Archived multilevel campaigns

The following figures are rendered from separately executed numerical campaigns. The main local meshes are material-fitted P4 with trace degree 2; a separate unfitted-local campaign uses the same degrees and exact material integration. Black/outlined edges show the actual macro geometry. No field values are averaged across macro interfaces.

In [ ]:
if HISTORICAL_REPLAY:
    unfitted = json.loads((root / "examples/results/unfitted/mhm-fitted-r16.json").read_text())
    [(row["setting"], row["delta"], row["gradient_l2"]) for row in unfitted]
    for name in ("fields-fitted-r16", "perturbation-fitted-r16", "publication-comparison",
                 "normal-flux-validation", "interface-flux-traces"):
        display(Image(filename=str(root / f"docs/figures/unfitted/{name}.png")))


## What is verified

Figure 5 of L10 uses an absolute broken-gradient L2 error, without the material weight. The five explicitly printed S1 errors agree with PyMHM within 0.553%; S0 differs by 0.264%. The S2 curve is not fully reproduced: at the largest perturbation our error is 0.00189484, whereas the published curve is approximately 0.00263. Historical local degree and local mesh are not specified numerically, and the references differ (the paper uses a fine Galerkin field; this study uses a verified Fourier series).

At the same perturbation, keeping local triangles unfitted increases the S2 gradient error from 0.00189484 to 0.00473545 despite exact material integration. Thus integration, local approximation and skeleton resolution are independent choices. Across the 33 fitted-local solves, conservative skeleton balances are below 2e-16. This does not imply pointwise continuity of the plotted raw flux $-a\nabla p_h$, a discrete maximum principle or fine-cell conservation. The signed component profiles preserve both incident material traces and actual macroface intersections.


## Independent DOLFINx/UFL operator and field verification

The full uncondensed P4/DG-P2 saddle is assembled independently with native UFL forms and solved by PETSc/MUMPS on the same 16-macro geometry. It agrees with PyMHM to floating-point precision for S1 and S2, including material-cut trace segments. This verifies the specified discretization; it does not remove the numerical discrepancy with the historical S2 curve or identify the article's unpublished local mesh.


In [ ]:
if HISTORICAL_REPLAY:
    native = json.loads((root / "examples/results/unfitted/ufl-verification.json").read_text())
    for row in native["rows"]:
        print({key: row[key] for key in ("setting", "delta", "native_dofs", "trace_dofs", "pressure_linf_difference", "trace_linf_difference", "gradient_l2")})


## Independent MSL smooth-problem controls

Six MSL MHM calculations use the same sine forcing and sixteen crisscross macrotriangles with continuous **local P1** pressures. The record identifies MSL Core, CG and MHM revisions, verified source URLs and executed driver digests. These controls are distinct from the local-P6 UFL systems and the high-order local-resolution campaign.

For trace degree zero and one segment, the absolute gradient error is 1.816304932, 1.799324401 and 1.795023635 at local refinements 8, 16 and 32. At refinement 32, degree zero/two segments and degree one/one segment give the same norm to rounding; degree one/two segments gives 0.262056788. The matching PyMHM/MSL fields differ by at most 2.14e-14 in nodal pressure and 1.64e-12 in a raw-flux component. This independently checks the coarse-trace plateau, without identifying a historical driver or reproducing Figure 2.


In [ ]:
if HISTORICAL_REPLAY:
    import hashlib

    msl_folder = root / "examples/results/unfitted/convergence/msl-smooth"
    msl = json.loads((msl_folder / "comparison.json").read_text())
    assert msl["accepted"] and len(msl["rows"]) == 6
    assert msl["numerical_library_source_modifications"] == []
    for row in msl["rows"]:
        archive = msl_folder / row["archive"]
        assert hashlib.sha256(archive.read_bytes()).hexdigest() == row["archive_sha256"]
    [
        {
            "case": row["case"],
            "absolute_gradient_error": row["reference_raw_flux_l2"],
            "pressure_field_difference_L2": row["pressure_field_difference_l2"],
            "raw_flux_field_difference_L2": row["raw_flux_field_difference_l2"],
        }
        for row in msl["rows"]
    ]


## Published h/p and contrast sweeps

The smooth test retains $p=\sin(2\pi x)\sin(2\pi y)$, its independently derived source $8\pi^2p$, homogeneous weak Dirichlet data and the stated sixteen-triangle crisscross partition. The declared local space is P8, with independent refinements 16, 24 and 32; it is not inferred to be the historical local solve. The plotted quantity is the **absolute broken-gradient norm**, without squaring, material weighting or a reference denominator.

The vector extraction preserves the 51 markers of Figures 2, 3 and 7 and their graphical intervals. Figure 2 places its coarse ell=0 value about 0.117304 at H=0.5, whereas Figure 3 places about 1.91044 at ell=0, H=Hcal. These nominally common labels do not identify equal numerical ordinates. The macro diameter and maximum face length are 1/2; the diagonal faces have length sqrt(2)/4. Equal subdivision of every face gives nominal H=1/(2s), while diagonal subfaces remain shorter. This declared bisection convention does not identify the historical input meshes or an unreported abscissa convention. The calculated curves and both sets of original markers are shown without fitted horizontal or vertical adjustments.

The Figure-7 contrast control explicitly selects ell=2 and delta=1/6 for S2. The Figure-7 caption does not restate those parameters. Material-fitted P4 locals at refinements 8 and 16, and the independent Fourier reference controls, test that declared configuration.


In [ ]:
if HISTORICAL_REPLAY:
    convergence_root = root / "examples/results/unfitted/convergence"
    convergence = json.loads((convergence_root / "comparison.json").read_text())
    published_path = root / "examples/results/unfitted/published-convergence.json"
    assert hashlib.sha256(published_path.read_bytes()).hexdigest() == convergence["published_sha256"]
    for name, expected_sha in convergence["acquisition_sha256"].items():
        path = convergence_root / name
        assert hashlib.sha256(path.read_bytes()).hexdigest() == expected_sha
        acquired = json.loads(path.read_text())
        assert acquired["complete"] and not acquired["source_changed_during_run"]
        for row in acquired["cases"]:
            archive = path.parent / row["archive"]
            assert hashlib.sha256(archive.read_bytes()).hexdigest() == row["archive_sha256"]
        for batch in acquired.get("acquisition_batches", []):
            assert hashlib.sha256((path.parent / batch["record"]).read_bytes()).hexdigest() == batch["record_sha256"]
    print("Acquired configurations:", list(convergence["acquisition_sha256"]))
    for figure in (2, 3, 7):
        rows = [row for row in convergence["printed_comparisons"] if row["figure"] == figure]
        print("Figure", figure, "markers", len(rows), "within graphical intervals",
              sum(row["inside_graphical_interval"] for row in rows))
        for row in rows:
            print(row["configuration"], row["computed_absolute_gradient_error"],
                  row["published_absolute_gradient_error"], row["published_graphical_interval"])


## Independent local-resolution controls

The volume differences below hold the skeletal space fixed and integrate both acquired pressures and gradients on their common physical subdivision. These differences are distinct from comparing two scalar exact-error norms. Orders 9 and 11 are recorded separately; no interpolation transfers the broken field across macrofaces. The local-resolution increments are measurements, not certified bounds for exact local solves.


In [ ]:
if HISTORICAL_REPLAY:
    for row in convergence["smooth_physical_local_differences"]:
        for name, expected_sha in row["record_sha256"].items():
            path = convergence_root / "local-resolution" / name
            assert hashlib.sha256(path.read_bytes()).hexdigest() == expected_sha
            check = json.loads(path.read_text())
            assert not check["source_changed_during_run"] and check["fields"] == row["fields"]
        print(row["name"], row["local_refinements"], row["physical_difference_by_quadrature"])
    for row in convergence["contrast_local_controls"]:
        print(row)
    for name in ("trace-h-convergence", "trace-p-convergence", "contrast-convergence"):
        display(Image(filename=str(root / f"docs/figures/unfitted/{name}.png")))


## Fixed-trace aligned local-refinement endpoints

Two additional controls keep ell=0 or ell=1 and 32 trace segments, and refine only the continuous local P8 pressure from r32 to r64. Both local boundaries resolve the trace breakpoints. The table records differences of the actual fields, separately from their absolute errors against the manufactured solution. These increments are not certified exact-local-error bounds.


In [ ]:
if HISTORICAL_REPLAY:
    endpoints = json.loads((convergence_root / 'smooth-p8-r64-endpoints.json').read_text())
    assert endpoints['complete'] and not endpoints['source_changed_during_run']
    assert [r['name'] for r in endpoints['cases']] == ['ell0-s32', 'ell1-s32']
    coarse = {r['name']:r for r in json.loads((convergence_root / 'smooth-p8-r32.json').read_text())['cases']}
    for row in endpoints['cases']:
        with (convergence_root / row['archive']).open('rb') as stream:
            assert hashlib.file_digest(stream, 'sha256').hexdigest() == row['archive_sha256']
        expected = [dict(name=coarse[row['name']]['archive'], sha256=coarse[row['name']]['archive_sha256']), dict(name=row['archive'], sha256=row['archive_sha256'])]
        print(row['name'], 'r64 absolute errors', row['norms'])
        for order in (9,11):
            path = convergence_root / 'local-resolution' / f"{row['name']}-r32-r64-q{order}.json"
            check = json.loads(path.read_text())
            assert check['fields'] == expected and check['quadrature_order'] == order
            assert not check['source_changed_during_run']
            print('physical difference', order, check['norms'])
